# Python Execution Models: Synchronous, Asynchronous, Multi-threading & Parallel Processing
**Lecture 14: Concurrency vs. Parallelism, Threading, Multiprocessing, AsyncIO & The Python GIL**  
*Last Updated: 23 Oct, 2026*

---

### Introduction: Efficiency in Python
In modern software and AI engineering, understanding how Python executes code is critical for writing fast, scalable, and responsive applications.

Python offers four primary execution paradigms:
1. **Synchronous (Sequential):** One waiter, one order at a time (Sequential blocking on a **single thread**).
2. **Asynchronous (`asyncio`):** One waiter who takes orders and returns later (Cooperative non-blocking Event Loop on a **single thread**).
3. **Multi-threading (`threading`):** Multiple waiters working inside the same restaurant (Preemptive concurrency on **multiple threads** within one process).
4. **Parallel Processing (`multiprocessing`):** Multiple chefs in separate kitchens (True parallelism across **multiple processes** on separate CPU cores).

> 💡 **CRITICAL ARCHITECTURAL INSIGHT:**  
> **Both Synchronous and Asynchronous functions run on a SINGLE THREAD!**  
> Neither creates new OS threads or processes. Asynchronous programming achieves high performance not through parallel hardware execution, but through intelligent non-blocking waiting via an **Event Loop**.


---
## 1. Synchronous — One Waiter, One Order at a Time

### Real-World Analogy: The Single-Threaded Waiter
Imagine a hotel has **one waiter**:
1. The waiter takes the **tea** order.
2. The waiter **waits** until the tea is prepared and served.
3. Only then does the waiter take the **biryani** order.
4. After that, the waiter handles the **coffee** order.

The orders are handled strictly one after another:
- **Tea:** 5 min
- **Biryani:** 10 min
- **Coffee:** 3 min
- **Total:** $5 + 10 + 3 = 18\text{ minutes}$

### How it Works in Python
In standard synchronous Python: **A function finishes completely before the next function call starts.** Every line of code blocks the thread until it finishes.

### Synchronous Architecture Diagram
```text
Python Program Starts (Single Thread)
          │
          ▼
Task 1: make_tea() ──────────► Prepare tea · Wait 2 seconds
                                       │
                                       ▼
                              Blocking / Waiting
                                       │
                                       ▼
                             Coffee cannot start yet!
                                       │
                                       ▼
Task 2: make_coffee() ───────► Prepare coffee · Wait 1 second
          │
          ▼
Program Completed (Total Time: ~3 seconds)
```

> **Thread Architecture:** Runs on a **single thread** (`MainThread`). While waiting for `time.sleep()`, the entire thread is frozen and cannot execute any other Python code.


In [1]:
import time
import threading

def make_tea():
    print(f"Making tea...     | Thread: {threading.current_thread().name}")
    time.sleep(2)  # Wait for 2 seconds (Blocking)
    print("Tea is ready!")

def make_coffee():
    print(f"Making coffee...  | Thread: {threading.current_thread().name}")
    time.sleep(1)  # Wait for 1 second (Blocking)
    print("Coffee is ready!")

start = time.time()

# Execute tasks one by one (Synchronously)
make_tea()
make_coffee()

print(f"All tasks completed in: {time.time() - start:.2f} seconds")


Making tea...     | Thread: MainThread
Tea is ready!
Making coffee...  | Thread: MainThread
Coffee is ready!
All tasks completed in: 3.00 seconds


---
## 2. Asynchronous (`asyncio`) — One Waiter Who Takes Orders and Returns Later

### Real-World Analogy: The Smart Async Waiter
Imagine the same waiter, but this time the waiter works like this:
1. Sends the **tea** order to the kitchen.
2. Instead of waiting idly, immediately takes the **biryani** order.
3. Sends the **biryani** order to the kitchen and immediately takes the **coffee** order.
4. Comes back to serve each order when it is notified that the dish is ready!

This is the foundation of **asynchronous programming**. When a task reaches an operation it can `await` (like an I/O wait, API response, or timer), the function pauses and **yields control back to the Event Loop**, allowing another task to make progress.

### ⚠️ IMPORTANT: Async Also Runs on a SINGLE THREAD!
> **Key Rule:** **BOTH Asynchronous and Synchronous code run on a SINGLE THREAD!**  
> Async does **NOT** spawn extra threads, and it does **NOT** use multiple CPU cores.  
> It achieves concurrency through a single-threaded **Event Loop** that rapidly switches between paused tasks ("coroutines") during waiting periods.

### Asynchronous Architecture Diagram
```text
Program starts
          │
          ▼
┌───────────────────────────────────────────────────────────────────┐
│                         Async Event Loop                          │
│                    (Runs on a SINGLE THREAD)                      │
│                Schedules and manages coroutines                   │
│                                                                   │
│       Task 1: Tea          Task 2: Biryani        Task 3: Coffee  │
│       Wait 3 seconds       Wait 3 seconds         Wait 3 seconds  │
│              │                    │                     │         │
│              └────────────► Waiting with await ◄────────┘         │
│                                                                   │
│      The Event Loop can run another task while one waits!         │
└───────────────────────────────────────────────────────────────────┘
                                  │
                                  ▼
                All orders completed in ~3 seconds!
```

**Result:** All three simulated waits overlap, so the program takes **about 3 seconds** instead of 9 seconds!


In [2]:
import asyncio
import time
import threading

async def order(food):
    print(f"Ordered: {food:<7} | Thread: {threading.current_thread().name}")
    await asyncio.sleep(3)  # Non-blocking simulated wait for kitchen
    print(f"Ready:   {food:<7} | Thread: {threading.current_thread().name}")

async def main():
    # Run all three orders concurrently inside the Single-Threaded Event Loop
    await asyncio.gather(
        order("Tea"),
        order("Biryani"),
        order("Coffee")
    )

start = time.time()
asyncio.run(main())
print(f"Total time: {time.time() - start:.2f} seconds (Single Thread Concurrency!)")


Ordered: Tea     | Thread: MainThread
Ordered: Biryani | Thread: MainThread
Ordered: Coffee  | Thread: MainThread
Ready:   Tea     | Thread: MainThread
Ready:   Biryani | Thread: MainThread
Ready:   Coffee  | Thread: MainThread
Total time: 3.01 seconds (Single Thread Concurrency!)


---
## 3. What is Multi-threading?

**Multi-threading** means running **multiple threads within the same process**.

- A **thread** is a smaller unit of execution inside a process.
- **Shared Memory:** Threads share the same memory and resources of their parent process.
- **Multiple Workers:** Unlike Async (which has 1 worker switching tasks), Multi-threading provides multiple execution workers managed by the OS.

### Real-World Example
Imagine you are working in a restaurant:
- **Thread 1:** Takes an order.
- **Thread 2:** Prepares a bill.
- **Thread 3:** Answers a phone call.

While one task is waiting, another can make progress. This is particularly useful for tasks involving waiting, such as:
- API calls
- Downloading files
- Reading files
- Database queries


In [3]:
import threading
import time

def task(name):
    print(f"{name} started")
    time.sleep(2)  # Simulates waiting for an API response
    print(f"{name} completed")

start = time.time()

t1 = threading.Thread(target=task, args=("Task A",))
t2 = threading.Thread(target=task, args=("Task B",))
t3 = threading.Thread(target=task, args=("Task C",))

t1.start()
t2.start()
t3.start()

t1.join()
t2.join()
t3.join()

print(f"Total time: {time.time() - start:.2f} seconds")


Task A started
Task B started
Task C started
Task A completed
Task B completed
Task C completed
Total time: 2.00 seconds


### Expected Output (order may vary):
```text
Task A started
Task B started
Task C started
Task A completed
Task B completed
Task C completed
Total time: 2.00 seconds
```

### Why does this take approximately 2 seconds rather than 6 seconds?
All three threads can wait at the same time.

### Multi-threading Timeline
Each block represents a 2-second wait:

```text
Sequential Timeline (Total: ~6.0s):
[ Task A: 2s Wait ] ───────> [ Task B: 2s Wait ] ───────> [ Task C: 2s Wait ]
|___________________|______________________|______________________|
0s                  2s                     4s                     6s

Multi-Threading Timeline (Total: ~2.0s):
Task A: [====== 2.0s Waiting / I/O ======] -> Completed
Task B: [====== 2.0s Waiting / I/O ======] -> Completed
Task C: [====== 2.0s Waiting / I/O ======] -> Completed
        +────────────────────────────────+
        0s                              2.0s
```

- **Task A:** Waiting / I/O
- **Task B:** Waiting / I/O
- **Task C:** Waiting / I/O

**All three waits overlap: approximately 2 seconds total.**

> **Important:** In standard CPython builds, the **Global Interpreter Lock (GIL)** generally prevents multiple threads from executing Python bytecode simultaneously. However, threads can still overlap I/O operations effectively because CPython releases the GIL during blocking I/O (like `time.sleep`, file access, or socket communication).


---
## 4. What is Parallel Processing?

**Parallel processing** means executing multiple tasks simultaneously using **multiple processes**, often on different CPU cores.

Each process has its own Python interpreter and separate memory space.

### Real-World Example
Imagine four workers, each with their own workstation:
- **Worker 1:** Calculates numbers in Task A.
- **Worker 2:** Calculates numbers in Task B.
- **Worker 3:** Calculates numbers in Task C.
- **Worker 4:** Calculates numbers in Task D.

All four workers can perform CPU-intensive calculations at the same time, provided sufficient CPU cores and resources are available.


In [4]:
import multiprocessing
import os
import time

def addition(a, b):
    print(f"Addition running | PID: {os.getpid()}")
    result = a + b
    print(f"Addition result: {result}")

def subtraction(a, b):
    print(f"Subtraction running | PID: {os.getpid()}")
    result = a - b
    print(f"Subtraction result: {result}")

def multiplication(a, b):
    print(f"Multiplication running | PID: {os.getpid()}")
    result = a * b
    print(f"Multiplication result: {result}")

if __name__ == "__main__":
    # Get inputs (demonstrated with sample values for notebook)
    num1 = 10.0
    num2 = 5.0
    print(f"Inputs: num1 = {num1}, num2 = {num2}\n")

    start = time.time()

    # Create separate processes
    p1 = multiprocessing.Process(
        target=addition, args=(num1, num2)
    )

    p2 = multiprocessing.Process(
        target=subtraction, args=(num1, num2)
    )

    p3 = multiprocessing.Process(
        target=multiplication, args=(num1, num2)
    )

    # Start all processes
    p1.start()
    p2.start()
    p3.start()

    # Wait for all processes to finish
    p1.join()
    p2.join()
    p3.join()

    print(f"\nTotal execution time: {time.time() - start:.2f} seconds")


Inputs: num1 = 10.0, num2 = 5.0

Addition running | PID: 15468
Addition result: 15.0
Subtraction running | PID: 7748
Subtraction result: 5.0
Multiplication running | PID: 6300
Multiplication result: 50.0

Total execution time: 0.29 seconds


---
## 5. Key Differences

| Feature | Synchronous | Asynchronous (`asyncio`) | Multi-threading | Parallel Processing |
| :--- | :--- | :--- | :--- | :--- |
| **Python module** | Built-in | `asyncio` | `threading` | `multiprocessing` |
| **Execution unit** | Sequential calls | Coroutines | Thread | Process |
| **Thread Count** | **Single Thread** | **Single Thread** | **Multiple Threads** | **Multiple Processes** |
| **Memory** | Shared process memory | Shared process memory | Shared within a process | Separate process memory by default |
| **Multiple CPU cores** | Uses 1 CPU core | Uses 1 CPU core | Limited by CPython GIL | Can execute CPU-bound code across multiple cores |
| **Creating workers** | No extra workers | Lightweight coroutines | Generally lightweight | Generally more resource-intensive |
| **Best suited for** | Simple sequential logic | Highly concurrent I/O (thousands of requests) | I/O-bound tasks | CPU-bound tasks |
| **Communication** | Direct variables | `asyncio.Queue` / events | Shared objects (needs `Lock`) | Queues, pipes, or IPC mechanisms |
| **Example** | Simple scripts, linear flows | Web scrapers, chat servers, API gateways | Multiple API requests, file downloads | Image processing, ML training, heavy math |


---
## 6. Understand the Difference Visually

### 1. Synchronous vs. Asynchronous (Both on a SINGLE THREAD!)
```text
Synchronous (1 Thread, Blocking):
Task A: [======== Work ========]───► Task B: [======== Work ========] (Sequential)

Asynchronous (1 Thread, Event Loop with Cooperative Switching):
Task A: [Start] ──(await)──► Task B: [Start] ──(await)──► Task A: [Done] ──► Task B: [Done]
              (While Task A awaits I/O, Task B runs on the SAME single thread!)
```

---

### 2. Multi-threading (One Process, Multiple Threads, Shared Memory)
```text
┌─────────────────────────────────────────────────────────────┐
│                       One Process                           │
│                                                             │
│    [ Thread 1 ]         [ Thread 2 ]         [ Thread 3 ]   │
│         │                    │                    │         │
│         └────────────────────┼────────────────────┘         │
│                              ▼                              │
│                    ┌──────────────────┐                     │
│                    │  Shared Memory   │                     │
│                    └──────────────────┘                     │
└─────────────────────────────────────────────────────────────┘
```

---

### 3. Parallel Processing (Multiple Processes, Separate Memory Across CPU Cores)
```text
┌────────────────────┐   ┌────────────────────┐   ┌────────────────────┐   ┌────────────────────┐
│     Process 1      │   │     Process 2      │   │     Process 3      │   │     Process 4      │
│                    │   │                    │   │                    │   │                    │
│  ┌──────────────┐  │   │  ┌──────────────┐  │   │  ┌──────────────┐  │   │  ┌──────────────┐  │
│  │   Separate   │  │   │  │   Separate   │  │   │  │   Separate   │  │   │  │   Separate   │  │
│  │    Memory    │  │   │  │    Memory    │  │   │  │    Memory    │  │   │  │    Memory    │  │
│  └──────────────┘  │   │  └──────────────┘  │   │  └──────────────┘  │   │  └──────────────┘  │
│         │          │   │         │          │   │         │          │   │         │          │
└─────────┼──────────┘   └─────────┼──────────┘   └─────────┼──────────┘   └─────────┼──────────┘
          ▼                        ▼                        ▼                        ▼
     [ CPU Core 1 ]           [ CPU Core 2 ]           [ CPU Core 3 ]           [ CPU Core 4 ]
```

**Key Takeaway:** Processes can execute simultaneously on separate CPU cores.


---
## 7. When Should You Use Each One?

Here are practical examples from **AI engineering** and **software development**:

### 1. When to use Synchronous:
- Simple command-line tools and sequential data-loading scripts.
- Tasks where step $N+1$ strictly depends on step $N$, and there is no I/O waiting time to optimize.

### 2. When to use Asynchronous (`asyncio`):
- Modern high-concurrency web backends (e.g., **FastAPI**, **Aiohttp**).
- Calling hundreds or thousands of LLM APIs concurrently (e.g. streaming prompts, multi-agent orchestrations).
- Web scrapers, live WebSocket connections, and chat servers.

### 3. When to use Multi-threading (`threading`):
- Legacy libraries that block and do not support `async`/`await`.
- GUI applications (e.g., Tkinter, PyQt) where long tasks run in a background thread to prevent UI freezing.
- Standard I/O operations (file reading, database calls).

### 4. When to use Parallel Processing (`multiprocessing`):
- 📊 Processing large datasets (Pandas, Polars, PyArrow partitions).
- 🧮 Performing independent mathematical calculations.
- 🖼️ CPU-heavy image transformations & video processing.
- ⚙️ Running computationally intensive Python functions (e.g., tokenization, feature extraction).

---

### Real-World AI Engineering Pipeline Example: Document Processing
In a production document-processing pipeline:
- You use **multiprocessing** for CPU-heavy tasks: converting PDF pages into high-resolution images, OCR tensor math, and local token crunching.
- You use **asyncio or multi-threading** to download source documents from cloud storage (AWS S3) and call remote LLM APIs concurrently without blocking!

> **Rule of Thumb:**  
> - **Waiting on remote networks / APIs?** $\longrightarrow$ **AsyncIO / Multi-threading (Single process)**  
> - **Crunching local CPU math / pixels / data?** $\longrightarrow$ **Multiprocessing (Multiple CPU cores)**


---
## 8. Summary Checklist for Students

- [x] **Single-Threaded Execution:** Both **Synchronous** and **Asynchronous (`asyncio`)** run on a **SINGLE THREAD**.
- [x] **Synchronous:** Blocks execution until each task finishes sequentially (Waiter waits for tea before taking biryani order).
- [x] **Asynchronous (`asyncio`):** Non-blocking Event Loop cooperatively switches tasks during `await` pauses (Waiter orders tea, takes biryani order while tea brews).
- [x] **Multi-threading (`threading`):** Multiple threads within one process, sharing memory; great for I/O-bound tasks.
- [x] **Parallel Processing (`multiprocessing`):** Multiple independent processes with separate memory spaces running on separate CPU cores; essential for CPU-bound tasks.
- [x] **CPython GIL:** Restricts Python bytecode execution to one thread at a time within a single process; bypassed completely by `multiprocessing`.
- [x] **Windows Multiprocessing Guard:** Always use `if __name__ == '__main__':` on Windows and macOS.

---
**Next Class:** Advanced Asynchronous Programming (`asyncio.gather`, `asyncio.Queue`, and Task Cancellation).
